In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download
import os
pd.set_option('display.max_columns', None)

In [2]:
caminho_diesel = hf_hub_download(
    repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
    filename="interim/diesel/diesel_deflacionado.parquet",
    repo_type="dataset"
)


caminho_safra = hf_hub_download(
repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
filename="interim/safra/producao_feijao_tons.parquet",
repo_type="dataset"
)

caminho_precos = hf_hub_download(
    repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
    filename="raw/painel_feijao_deflacionado.csv",
    repo_type="dataset"
)



caminho_repasse = hf_hub_download(
    repo_id="pbf-feijao-mecai-usp/bf-feijao-dados",
    filename="raw/Painel_Unificado_2004_2026_Deflacionado.xlsx",
    repo_type="dataset"
)

datasets = {
    "diesel": caminho_diesel,
    "safra": caminho_safra,
    "precos": caminho_precos,
    "bolsa_familia": caminho_repasse
}

In [3]:
datasets_carregados = {}

for nome, caminho in datasets.items():
    if caminho.endswith(".parquet"):
        df_temp = pd.read_parquet(caminho)
    elif  caminho.endswith(".csv"):
        df_temp = pd.read_csv(caminho)
    elif caminho.endswith(".xlsx"):
        df_temp = pd.read_excel(caminho)
    else:
        continue
    datasets_carregados[nome] = df_temp

In [4]:
df_precos = datasets_carregados["precos"]
df_safra = datasets_carregados["safra"]
df_diesel = datasets_carregados["diesel"]
df_repasse = datasets_carregados["bolsa_familia"]

### Tratamentos indiduais das tabelas para construir o dataset unificado

In [5]:
# dic_estado_sigla = {
#     # Norte
#     "acre": "AC",
#     "amapa": "AP",
#     "amazonas": "AM",
#     "para": "PA",
#     "rondonia": "RO",
#     "roraima": "RR",
#     "tocantins": "TO",
    
#     # Nordeste
#     "alagoas": "AL",
#     "bahia": "BA",
#     "ceara": "CE",
#     "maranhao": "MA",
#     "paraiba": "PB",
#     "pernambuco": "PE",
#     "piaui": "PI",
#     "rio_grande_do_norte": "RN",
#     "sergipe": "SE",
    
#     # Centro-Oeste
#     "distrito_federal": "DF",
#     "goias": "GO",
#     "mato_grosso": "MT",
#     "mato_grosso_do_sul": "MS",
    
#     # Sudeste
#     "espirito_santo": "ES",
#     "minas_gerais": "MG",
#     "rio_de_janeiro": "RJ",
#     "sao_paulo": "SP",
    
#     # Sul
#     "parana": "PR",
#     "rio_grande_do_sul": "RS",
#     "santa_catarina": "SC"
# }

In [6]:
# Tratamentos de SAFRA

data_safra = df_safra["ano"].astype(str) + "-" + df_safra["mes"].astype(str) + "-01"
df_safra["data_referencia"] = pd.to_datetime(data_safra)

df_safra["producao_mensal_nacional"] = (
    df_safra.groupby(["ano", "mes"])["quantidade_tons"].transform('sum')
)
df_safra_agrupado = df_safra.groupby([
    "data_referencia",
    "producao_mensal_nacional"
    ])["quantidade_tons"].sum().reset_index()

# df_safra_agrupado["uf"] = df_safra_agrupado["uf"].map(dic_estado_sigla)

# df_safra_agrupado.rename(columns={"quantidade_tons": "produção_feijao_tons"}, inplace=True )

df_safra_agrupado.drop(columns=["quantidade_tons"], inplace=True)

display(df_safra_agrupado.head())
len(df_safra_agrupado)

,data_referencia,producao_mensal_nacional
0,2006-09-01,3394438
1,2006-10-01,3377163
2,2006-11-01,3411554
3,2006-12-01,3406478
4,2007-01-01,4120806


239

In [7]:
dic_capital_formatada = {
    # Norte
    "RIO BRANCO": "Rio Branco",
    "MACAPA": "Macapá",
    "MANAUS": "Manaus",
    "BELEM": "Belém",
    "PORTO VELHO": "Porto Velho",
    "BOA VISTA": "Boa Vista",
    "PALMAS": "Palmas",
    
    # Nordeste
    "MACEIO": "Maceió",
    "SALVADOR": "Salvador",
    "FORTALEZA": "Fortaleza",
    "SAO LUIS": "São Luís",
    "JOAO PESSOA": "João Pessoa",
    "RECIFE": "Recife",
    "TERESINA": "Teresina",
    "NATAL": "Natal",
    "ARACAJU": "Aracaju",
    
    # Centro-Oeste
    "BRASILIA": "Brasília",
    "GOIANIA": "Goiânia",
    "CUIABA": "Cuiabá",
    "CAMPO GRANDE": "Campo Grande",
    
    # Sudeste
    "VITORIA": "Vitória",
    "BELO HORIZONTE": "Belo Horizonte",
    "RIO DE JANEIRO": "Rio de Janeiro",
    "SAO PAULO": "São Paulo",
    
    # Sul
    "CURITIBA": "Curitiba",
    "PORTO ALEGRE": "Porto Alegre",
    "FLORIANOPOLIS": "Florianópolis"
}

In [8]:
# Tratamentos do DIESEL
df_diesel["data_referencia"] = df_diesel["data_da_coleta"].dt.to_period("M").dt.to_timestamp()
df_diesel["year"] = df_diesel["data_da_coleta"].dt.year
df_diesel["month"] = df_diesel["data_da_coleta"].dt.month
capitais = ["ARACAJU", "BELEM", "BELO HORIZONTE", "BRASILIA", "CURITIBA", "FORTALEZA", "GOIANIA", "JOAO PESSOA", "NATAL", "PORTO ALEGRE", "RECIFE", "RIO DE JANEIRO", "SALVADOR", "SAO PAULO", "VITORIA", "FLORIANOPOLIS"]

df_diesel = df_diesel[df_diesel['municipio'].isin(capitais)]

df_diesel_agrupado = df_diesel.pivot_table(
    index=["data_referencia", "municipio"],
    columns="produto",
    values=["valor_de_venda", "valor_de_venda_real"],
    aggfunc="mean"
)

df_diesel_agrupado.columns = [f"{andar0}_{andar1}" for andar0, andar1 in df_diesel_agrupado.columns]
df_diesel_agrupado = df_diesel_agrupado.reset_index()

# cols_venda = ['valor_de_venda_real_DIESEL', 'valor_de_venda_real_diesel_s50', 'valor_de_venda_real_diesel_s10']
# cols_venda_existentes = [col for col in cols_venda if col in df_diesel_agrupado.columns]
# df_diesel_agrupado['venda_real_diesel_media_geral'] = df_diesel_agrupado[cols_venda_existentes].astype(float).mean(axis=1, skipna=True)

# cols_nom = ['valor_de_venda_DIESEL', 'valor_de_venda_diesel_s50', 'valor_de_venda_diesel_s10']
# cols_nom_existentes = [col for col in cols_nom if col in df_diesel_agrupado.columns]
# df_diesel_agrupado['valor_venda_diesel_media_geral'] = df_diesel_agrupado[cols_nom_existentes].astype(float).mean(axis=1, skipna=True)


cols_real = df_diesel_agrupado.filter(like='valor_de_venda_real_DIESEL').columns
df_diesel_agrupado['venda_real_diesel_media_geral'] = df_diesel_agrupado[cols_real].astype(float).mean(axis=1, skipna=True)

cols_nom = df_diesel_agrupado.filter(regex='^valor_de_venda_DIESEL').columns
df_diesel_agrupado['valor_venda_diesel_media_geral'] = df_diesel_agrupado[cols_nom].astype(float).mean(axis=1, skipna=True)


df_diesel_agrupado.rename(
    columns={
        'municipio': 'capital',
        'valor_de_venda_real_DIESEL': 'venda_real_diesel_comum',
        'valor_de_venda_DIESEL': 'valor_venda_diesel_comum'
    }, 
    inplace=True
)

df_diesel_agrupado["capital"] = df_diesel_agrupado["capital"].map(dic_capital_formatada)


display(df_diesel_agrupado.head())
len(df_diesel_agrupado)

,data_referencia,capital,valor_venda_diesel_comum,valor_de_venda_DIESEL S10,valor_de_venda_DIESEL S50,venda_real_diesel_comum,valor_de_venda_real_DIESEL S10,valor_de_venda_real_DIESEL S50,venda_real_diesel_media_geral,valor_venda_diesel_media_geral
0,2004-05-01,Aracaju,1.335657,NaN,NaN,4.450634,NaN,NaN,4.450634,1.335657
1,2004-05-01,Belém,1.427871,NaN,NaN,4.757907,NaN,NaN,4.757907,1.427871
2,2004-05-01,Belo Horizonte,1.369720,NaN,NaN,4.564138,NaN,NaN,4.564138,1.369720
3,2004-05-01,Brasília,1.452098,NaN,NaN,4.838634,NaN,NaN,4.838634,1.452098
4,2004-05-01,Curitiba,1.380254,NaN,NaN,4.599237,NaN,NaN,4.599237,1.380254


4216

In [9]:
# Tratamentos de PREÇO DO FEIJÂO

df_precos["data_referencia"] = pd.to_datetime(df_precos["periodo"], format="%m/%Y")
df_precos.drop(columns=["periodo","ano","mes", "t"], inplace=True)
df_precos.rename(columns={"preco_nominal": "preco_nominal_feijao",
                          "preco_real":"preco_real_feijao",
                          "log_preco_real":"log_preco_feijao_real",
                          "var_pct_real":"var_pct_ipca_real",
                          "log_preco_nominal":"log_preco_feijao_nominal"
                          }, inplace=True)
display(df_precos.head())
len(df_precos)

,capital_id,capital,uf,regiao,preco_nominal_feijao,preco_real_feijao,log_preco_feijao_real,ipca_var_pct,ipca_indice,ln_ipca_indice,var_pct_ipca_real,log_preco_feijao_nominal,data_referencia
0,1,Brasília,DF,Centro-Oeste,2.41,2.4100,0.879627,0.76,100.0000,0.693147,NaN,0.879627,2004-01-01
1,1,Brasília,DF,Centro-Oeste,2.40,2.3836,0.868612,0.69,100.6900,0.696591,-1.0954,0.875469,2004-02-01
2,1,Brasília,DF,Centro-Oeste,2.40,2.3724,0.863902,0.47,101.1632,0.698947,-0.4699,0.875469,2004-03-01
3,1,Brasília,DF,Centro-Oeste,2.44,2.4031,0.876760,0.37,101.5375,0.700806,1.2940,0.891998,2004-04-01
4,1,Brasília,DF,Centro-Oeste,2.47,2.4203,0.883891,0.51,102.0554,0.703372,0.7157,0.904218,2004-05-01


4336

In [10]:
# Tratamentos de REPASSE DO BOLSA FAMÍLIA

df_repasse["data_referencia"] = pd.to_datetime(df_repasse["periodo"], format="%Y%m")
df_repasse.rename(columns={
                           "codigo":"codigo_cidade",
                           "cidade":"capital",
                           "valor_repassado":"valor_repassado_pbf",
                           "valor_repassado_real":"valor_repassado_pbf_real",
                           "ln_valor_repassado_real":"ln_valor_repassado_pbf_real"
                           }, inplace=True)
df_repasse.drop(columns="periodo", inplace=True)
display(df_repasse.head())
len(df_repasse)

,codigo_cidade,capital,programa,qtd_familias,valor_repassado_pbf,ipca_indice,valor_repassado_pbf_real,ln_valor_repassado_pbf_real,data_referencia
0,150140,Belém,Bolsa Família,298,20865,100.000000,20865.00,9.945828,2004-01-01
1,150140,Belém,Bolsa Família,298,20865,100.690000,20722.02,9.938952,2004-02-01
2,150140,Belém,Bolsa Família,298,20865,101.163243,20625.08,9.934263,2004-03-01
3,150140,Belém,Bolsa Família,369,24380,101.537547,24010.82,10.086260,2004-04-01
4,150140,Belém,Bolsa Família,11026,686140,102.055388,672321.19,13.418491,2004-05-01


4352

In [11]:
# Alinhando todos os textos que serão indices do merge
def padronizar_texto(coluna_pandas):
    return (
        coluna_pandas
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
    )

df_precos['capital_join'] = padronizar_texto(df_precos['capital'])
df_repasse['capital_join'] = padronizar_texto(df_repasse['capital'])
df_diesel_agrupado['capital_join'] = padronizar_texto(df_diesel_agrupado['capital'])

df_precos['uf_join'] = df_precos['uf'].str.upper().str.strip()
# df_safra_agrupado['uf_join'] = df_safra_agrupado['uf'].str.upper().str.strip()

In [12]:
# unindo as tabelas
df_unificado = df_precos.merge(df_safra_agrupado, on= "data_referencia", how="left") \
    .merge(df_diesel_agrupado, on= ["data_referencia","capital"], how="left") \
    .merge(df_repasse, on= ["data_referencia","capital"], how="left")

In [13]:
# descartando colunas auxiliares
colunas_join_descartar = [col for col in df_unificado.columns if "_join" in col]
colunas_y_descartar = [col for col in df_unificado.columns if col.endswith("_y")]
df_unificado.drop(columns=colunas_join_descartar + colunas_y_descartar, inplace= True)
df_unificado.columns = df_unificado.columns.str.replace("_x","")

In [14]:
display(df_unificado.sample(5))
len(df_unificado)

,capital_id,capital,uf,regiao,preco_nominal_feijao,preco_real_feijao,log_preco_feijao_real,ipca_var_pct,ipca_indice,ln_ipca_indice,var_pct_ipca_real,log_preco_feijao_nominal,data_referencia,producao_mensal_nacional,valor_venda_diesel_comum,valor_de_venda_DIESEL S10,valor_de_venda_DIESEL S50,venda_real_diesel_comum,valor_de_venda_real_DIESEL S10,valor_de_venda_real_DIESEL S50,venda_real_diesel_media_geral,valor_venda_diesel_media_geral,codigo_cidade,programa,qtd_familias,valor_repassado_pbf,valor_repassado_pbf_real,ln_valor_repassado_pbf_real
3681,14,Natal,RN,Nordeste,5.23,2.4359,0.890316,0.25,214.7067,1.146471,-13.7673,1.654411,2017-03-01,3368982.0,3.095315,3.292614,NaN,4.900201,5.212545,NaN,5.056373,3.193964,240810,Bolsa Família,44787,6629589,3087742.31,14.942951
1085,5,São Paulo,SP,Sudeste,2.71,2.6914,0.990062,0.69,100.6900,0.696591,7.6560,0.996949,2004-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,355030,Bolsa Família,54039,3695745,3670419.11,15.115816
3776,14,Natal,RN,Nordeste,7.43,2.3129,0.838502,1.31,321.2479,1.438051,-4.6266,2.005526,2025-02-01,3396100.0,6.636316,6.658485,NaN,7.030717,7.054203,NaN,7.042460,6.647400,240810,Bolsa Família,75955,50849778,15828826.26,16.577343
73,1,Brasília,DF,Centro-Oeste,2.97,2.1734,0.776293,0.78,136.6512,0.861417,-0.4398,1.088562,2010-02-01,3716931.0,2.011213,NaN,NaN,5.010574,NaN,NaN,5.010574,2.011213,530010,Bolsa Família,71271,5897721,4315894.85,15.277815
3034,12,Fortaleza,CE,Nordeste,4.45,3.5242,1.259653,0.74,126.2691,0.816555,-2.7029,1.492904,2008-06-01,3480485.0,2.043850,NaN,NaN,5.510524,NaN,NaN,5.510524,2.043850,230440,Bolsa Família,148623,10621243,8411595.95,15.945122


4336

In [15]:
def polimento(self):
    
    colunas_duplicadas = [col for col in self.columns if col.endswith('_y')]
    self.drop(columns=colunas_duplicadas, inplace=True)
    self.columns = self.columns.str.replace('_x', '')

    return(self)


In [16]:
dataset_final = polimento(df_unificado)

In [17]:
dataset_final.columns

Index(['capital_id', 'capital', 'uf', 'regiao', 'preco_nominal_feijao',
       'preco_real_feijao', 'log_preco_feijao_real', 'ipca_var_pct',
       'ipca_indice', 'ln_ipca_indice', 'var_pct_ipca_real',
       'log_preco_feijao_nominal', 'data_referencia',
       'producao_mensal_nacional', 'valor_venda_diesel_comum',
       'valor_de_venda_DIESEL S10', 'valor_de_venda_DIESEL S50',
       'venda_real_diesel_comum', 'valor_de_venda_real_DIESEL S10',
       'valor_de_venda_real_DIESEL S50', 'venda_real_diesel_media_geral',
       'valor_venda_diesel_media_geral', 'codigo_cidade', 'programa',
       'qtd_familias', 'valor_repassado_pbf', 'valor_repassado_pbf_real',
       'ln_valor_repassado_pbf_real'],
      dtype='str')

In [18]:
# Nova ordenação das colunas seguindo a seguinte sequência: Tempo, Geografia, Alvo (Preço), Oferta (Safra), Custo (Diesel), Demanda (PBF)
ordem = [
    # 1. Chaves de Identificação (Tempo e Espaço)
    'data_referencia',
    'capital_id',
    'codigo_cidade',
    'capital',
    'uf',
    'regiao',
    
    # 2. Variável Dependente (O Preço do Feijão)
    'preco_nominal_feijao',
    'preco_real_feijao',
    'log_preco_feijao_nominal',
    'log_preco_feijao_real',
    
    # 3. Variável de Oferta (Safra)
    'producao_mensal_nacional',
    
    # 4. Variável de Demanda (Bolsa Família)
    'programa',
    'qtd_familias',
    'valor_repassado_pbf',
    'valor_repassado_pbf_real',
    'ln_valor_repassado_pbf_real',
    
    # 5. Variável de Custo/Logística (Diesel)
    'valor_venda_diesel_comum',
    'valor_de_venda_DIESEL S10',
    'valor_de_venda_DIESEL S50',
    'valor_venda_diesel_media_geral',
    'venda_real_diesel_comum',
    'valor_de_venda_real_DIESEL S10',
    'valor_de_venda_real_DIESEL S50',
    'venda_real_diesel_media_geral',
    
    # 6. Variáveis de Controle Macroeconômico (Inflação)
    'ipca_indice',
    'ipca_var_pct',
    'ln_ipca_indice',
    'var_pct_ipca_real'
]


In [19]:
# dataset final com as colunas reordenadas
dataset_final = dataset_final[ordem]

In [20]:
display(dataset_final.sample(10))

,data_referencia,capital_id,codigo_cidade,capital,uf,regiao,preco_nominal_feijao,preco_real_feijao,log_preco_feijao_nominal,log_preco_feijao_real,producao_mensal_nacional,programa,qtd_familias,valor_repassado_pbf,valor_repassado_pbf_real,ln_valor_repassado_pbf_real,valor_venda_diesel_comum,valor_de_venda_DIESEL S10,valor_de_venda_DIESEL S50,valor_venda_diesel_media_geral,venda_real_diesel_comum,valor_de_venda_real_DIESEL S10,valor_de_venda_real_DIESEL S50,venda_real_diesel_media_geral,ipca_indice,ipca_var_pct,ln_ipca_indice,var_pct_ipca_real
479,2021-05-01,2,520870,Goiânia,GO,Centro-Oeste,7.52,2.9512,2.017566,1.082212,2887118.0,Bolsa Família,36331,2256093,885394.70,13.693789,4.538308,4.605761,NaN,4.572034,6.035645,6.125353,NaN,6.080499,254.8121,0.83,1.266418,1.7480
715,2018-06-01,3,310620,Belo Horizonte,MG,Sudeste,3.69,1.6427,1.305626,0.496341,3388558.0,Bolsa Família,57466,8334941,3710584.35,15.126700,3.528102,3.562561,NaN,3.545332,5.338691,5.390834,NaN,5.364763,224.6261,1.26,1.177504,4.1133
2877,2017-12-01,11,280030,Aracaju,SE,Nordeste,4.08,1.8636,1.406097,0.622510,3291312.0,Bolsa Família,30948,4202364,1919531.86,14.467592,3.326455,3.481761,NaN,3.404108,5.164609,5.405735,NaN,5.285172,218.9265,0.44,1.159790,-2.8211
865,2008-05-01,4,330455,Rio de Janeiro,RJ,Sudeste,3.78,3.0158,1.329724,1.103865,3500162.0,Bolsa Família,125388,9334019,7446868.51,15.823304,2.001807,NaN,NaN,2.001807,5.437116,NaN,NaN,5.437116,125.3415,0.79,0.812447,-2.5873
2264,2012-01-01,9,431490,Porto Alegre,RS,Sul,2.88,1.8866,1.057790,0.634776,3527421.0,Bolsa Família,41447,4998148,3274118.96,15.001559,2.071624,NaN,NaN,2.071624,4.619969,NaN,NaN,4.619969,152.6563,0.56,0.926860,-0.5587
3191,2021-07-01,12,230440,Fortaleza,CE,Nordeste,7.76,3.0005,2.048982,1.098779,2740838.0,Bolsa Família,217600,12976534,5017571.90,15.428457,4.728353,4.743118,NaN,4.735736,6.195762,6.215110,NaN,6.205436,258.6218,0.96,1.277098,-1.9637
3844,2008-03-01,15,261160,Recife,PE,Nordeste,5.25,4.2449,1.658228,1.445718,3417865.0,Bolsa Família,109990,8515427,6885111.14,15.744872,1.843393,NaN,NaN,1.843393,5.074157,NaN,NaN,5.074157,123.6789,0.48,0.805041,-19.9845
1720,2011-11-01,7,410690,Curitiba,PR,Sul,2.69,1.7809,0.989541,0.577119,3525854.0,Bolsa Família,39306,4229856,2800285.11,14.845232,1.957323,NaN,NaN,1.957323,4.411450,NaN,NaN,4.411450,151.0509,0.52,0.920486,-0.5139
3941,2016-04-01,15,261160,Recife,PE,Nordeste,6.54,3.1659,1.877937,1.152437,3207448.0,Bolsa Família,106302,12799121,6195906.35,15.639399,NaN,3.059757,NaN,3.059757,NaN,5.034614,NaN,5.034614,206.5738,0.61,1.120288,4.0046
2142,2024-06-01,8,420540,Florianópolis,SC,Sul,9.21,2.9895,2.220290,1.095106,3216963.0,Bolsa Família,14848,9645200,3130735.07,14.956778,NaN,5.970000,NaN,5.970000,NaN,6.564909,NaN,6.564909,308.0810,0.20,1.406296,-7.0602


In [21]:
dataset_final.to_csv("dataset_econometria_feijao_pbf.csv")